In [73]:
import os 
from dotenv import load_dotenv

# Langchain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings

In [74]:
load_dotenv()

True

In [75]:
groq_key = os.getenv("GROQ_KEY")
jina_key = os.getenv("JINA_KEY")

In [76]:
DATA_FILE_PATH = os.path.join("data", "hr_policies.txt")

### Data Ingestion

In [77]:
DATA_FILE_PATH = os.path.join("data", "hr_policy.txt")

loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")
documents = loader.load()

print(f"Loaded file: {DATA_FILE_PATH}")
print(f"Number of documents loaded: {len(documents)}")
print(f"Total characters in document: {len(documents[0].page_content)}")
print("\n--- Preview of first 300 characters ---")
print(documents[0].page_content[:300])

Loaded file: data\hr_policy.txt
Number of documents loaded: 1
Total characters in document: 2598

--- Preview of first 300 characters ---
COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carr


### Langchain document

In [78]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [79]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


#### Splitting the data

In [80]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = text_splitter.split_documents(documents)
print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [81]:
len(chunks)

9

In [82]:
print("\n--- Splitting document into chunks ---")
print(chunks[0].page_content)


--- Splitting document into chunks ---
COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)


### Store data in vector DB

In [83]:
from langchain_community.embeddings import JinaEmbeddings

embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

print("EMB MODEL READY THE NAME IS ", embeddings_model.model_name)

EMB MODEL READY THE NAME IS  jina-embeddings-v2-base-en


In [84]:
import os

# Check what the kernel sees
print("Loaded Key:", os.environ.get("JINA_API_KEY"))

Loaded Key: jina_6aefc9ae4dba43b5b7ecf87bb84394d5GMnr1ZXtQ2IgJX0qVZPdjnTzBlVW


In [85]:
test_embedding = embeddings_model.embed_query("This is a test document.")

print("Embedding dimensions:", len(test_embedding))

RuntimeError: Insufficient account balance. Top up your account at https://jina.ai/api-dashboard/key-manager.

In [ ]:
embeddings_model = JinaEmbeddings(
    model_name="jina-embeddings-v2-base-en"
)

In [ ]:
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(chunks, embeddings_model)
print("Chunks added to vectorstore", vector_store.index.ntotal)

RuntimeError: Insufficient account balance. Top up your account at https://jina.ai/api-dashboard/key-manager.